In [1]:
# ==============================================================================
# תא 1: ייבוא ספריות והגדרת נתיבים
# ==============================================================================
import pandas as pd
import numpy as np
import re
import os
import time
import pickle
from sklearn.model_selection import cross_validate, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
import warnings
warnings.filterwarnings('ignore')

DATASET_PATH = 'C:/Users/yarin/Downloads/dataset.csv'
MODEL_PICKLE_PATH = 'movie_rating_pipeline.pkl'

In [2]:
# ==============================================================================
# תא 2: פונקציית prepare_data (מתוקן לדרישות המרצה - ללא אינפלציה)
# ==============================================================================
import numpy as np
import pandas as pd
import re

def prepare_data(df):
    df_clean = df.copy()

    expected_cols = ['startYear', 'budget', 'Country', 'Language', 'runtimeMinutes', 'genres', 'lead_actors_ids']
    for col in expected_cols:
        if col not in df_clean.columns:
            df_clean[col] = np.nan

    df_clean['startYear'] = pd.to_numeric(df_clean['startYear'], errors='coerce')
    decades = [1910, 1920, 1930, 1940, 1950, 1960, 1970, 1980, 1990, 2000, 2010, 2020]
    for dec in decades:
        df_clean[f'is_decade_{dec}'] = ((df_clean['startYear'] // 10) * 10 == dec).astype(int)
    df_clean['is_decade_unknown'] = df_clean['startYear'].isna().astype(int)

    def parse_advanced_budget(row):
        val = row.get('budget', np.nan)
        year = row.get('startYear', np.nan)
        country = str(row.get('Country', '')).lower()
        lang = str(row.get('Language', '')).lower()

        if pd.isna(val) or str(val).strip() in ['', 'unknown', 'nan', '\n']: return np.nan

        val_str = str(val).lower()
        match = re.search(r'\d+(\.\d+)?', val_str.replace(',', ''))
        if not match: return np.nan
        
        result = float(match.group())
        if result == 0: return np.nan

        has_explicit = False
        if 'billion' in val_str or 'b' in val_str: result *= 1e9; has_explicit = True
        elif 'million' in val_str or 'm' in val_str: result *= 1e6; has_explicit = True
        elif 'thousand' in val_str or 'k' in val_str: result *= 1e3; has_explicit = True

        if not has_explicit:
            if 0 < result <= 100: result *= 1e6  
            elif 100 < result <= 1000: result *= 1e3  

        is_indian = 'india' in country or any(l in lang for l in ['hindi', 'tamil', 'telugu', 'malayalam'])
        is_japanese = 'japan' in country or 'japanese' in lang

        if '₹' in val_str or 'inr' in val_str or (is_indian and '$' not in val_str and 'usd' not in val_str): result /= 80.0
        elif '¥' in val_str or 'jpy' in val_str or (is_japanese and '$' not in val_str and 'usd' not in val_str): result /= 150.0 
        elif '£' in val_str or 'gbp' in val_str: result *= 1.25  
        elif '€' in val_str or 'eur' in val_str: result *= 1.10  

        # הערה: חישוב האינפלציה/היוון הוסר לחלוטין מכאן.
        
        return result

    df_clean['budget_parsed'] = df_clean.apply(parse_advanced_budget, axis=1)

    top_langs = ['English', 'French', 'Hindi', 'Spanish', 'Italian', 'Japanese', 'Tamil', 'German', 'Telugu', 'Malayalam']
    def cat_lang(val):
        res = {f'is_lang_{l}': 0 for l in top_langs}; res.update({'is_lang_other': 0, 'is_lang_unknown': 0})
        val = str(val).strip()
        if val in ['Not Found', 'unknown', 'nan', '', 'none']: res['is_lang_unknown'] = 1; return pd.Series(res)
        text = val.lower(); found_any = False
        for l in top_langs:
            if l.lower() in text: res[f'is_lang_{l}'] = 1; text = text.replace(l.lower(), ''); found_any = True
        if len(re.sub(r'[^\w\s]', '', text).strip()) > 2 or not found_any: res['is_lang_other'] = 1
        return pd.Series(res)
    df_clean = pd.concat([df_clean, df_clean['Language'].apply(cat_lang)], axis=1)

    top_countries = ['United States', 'United Kingdom', 'India', 'France', 'Japan', 'Canada', 'Germany', 'Italy', 'Spain', 'Australia']
    def cat_country(val):
        res = {f'is_country_{c.replace(" ", "_")}': 0 for c in top_countries}; res.update({'is_other_country': 0, 'unknown_country': 0})
        val = str(val).strip()
        if val in ['Not Found', 'unknown', 'nan', '', 'none']: res['unknown_country'] = 1; return pd.Series(res)
        text = val.lower()
        for c in top_countries:
            if c.lower() in text: res[f'is_country_{c.replace(" ", "_")}'] = 1; text = text.replace(c.lower(), '')
        if len(re.sub(r'[^\w\s]', '', text).strip()) > 2: res['is_other_country'] = 1
        if sum(res.values()) == 0: res['unknown_country'] = 1
        return pd.Series(res)
    df_clean = pd.concat([df_clean, df_clean['Country'].apply(cat_country)], axis=1)

    if 'averageRating' in df_clean.columns: 
        df_clean['averageRating'] = pd.to_numeric(df_clean['averageRating'], errors='coerce')
    
    # === התיקון: שיטת הרשימה הלבנה (Whitelist) ===
    
    # 1. איסוף כל עמודות הקידוד שיצרנו (כולל is_other_country)
    dummy_cols = [col for col in df_clean.columns if col.startswith(('is_decade_', 'is_lang_', 'is_country_', 'is_other_country', 'unknown_country'))]
    
    # 2. עמודות הבסיס שה-Pipeline צריך להמשך
    base_pipeline_cols = ['runtimeMinutes', 'genres', 'lead_actors_ids', 'budget_parsed']
    
    # 3. איחוד לרשימה סופית וקשיחה של פיצ'רים
    final_features = base_pipeline_cols + dummy_cols
    
    # 4. אם קיים ציון (קובץ אימון), נוסיף אותו כדי שיופרד בהמשך.
    if 'averageRating' in df_clean.columns:
        final_features.append('averageRating')
        
    # החזרת DataFrame עם העמודות הרלוונטיות בלבד
    return df_clean[final_features]

In [3]:
# ==============================================================================
# תא 3: טרנספורמרים לומדים למניעת Data Leakage
# ==============================================================================
class GenreTargetEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, min_count=10, smoothing_m=20):
        self.min_count = min_count; self.smoothing_m = smoothing_m
    def fit(self, X, y):
        df_temp = pd.DataFrame({'genres_raw': X['genres'], 'rating': y})
        def clean_genre(val): return ",".join(sorted([g.strip().lower() for g in str(val).replace('[','').replace(']','').replace("'","").replace('"','').split(',') if g.strip()])) if pd.notna(val) else 'unknown'
        df_temp['clean_genres'] = df_temp['genres_raw'].apply(clean_genre)
        self.global_mean_ = y.mean()
        stats = df_temp.groupby('clean_genres')['rating'].agg(['mean', 'count'])
        stats = stats[stats['count'] >= self.min_count]
        stats['smoothed'] = ((stats['count'] * stats['mean']) + (self.smoothing_m * self.global_mean_)) / (stats['count'] + self.smoothing_m)
        self.genre_means_ = stats['smoothed'].to_dict()
        return self
    def transform(self, X):
        X_new = X.copy()
        def get_genre_score(val): return self.genre_means_.get(",".join(sorted([g.strip().lower() for g in str(val).replace('[','').replace(']','').replace("'","").replace('"','').split(',') if g.strip()])), self.global_mean_) if pd.notna(val) else self.global_mean_
        X_new['genre_target_encoded'] = X_new['genres'].apply(get_genre_score)
        return X_new.drop(columns=['genres'], errors='ignore')

class RuntimeHighPolynomial(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        temp_runtime = pd.to_numeric(X['runtimeMinutes'], errors='coerce')
        self.median_ = temp_runtime.median()
        if pd.isna(self.median_): self.median_ = 100 
        return self
    def transform(self, X):
        X_new = X.copy()
        temp_runtime = pd.to_numeric(X_new['runtimeMinutes'], errors='coerce').fillna(self.median_)
        X_new['runtimeMinutes'] = temp_runtime
        X_new['runtimeMinutes_sq'] = (temp_runtime / 100.0) ** 2
        X_new['runtimeMinutes_cu'] = (temp_runtime / 100.0) ** 3
        X_new['runtimeMinutes_qd'] = (temp_runtime / 100.0) ** 4
        return X_new

class BudgetLogWithMissingFlag(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.median_ = X['budget_parsed'].median()
        if pd.isna(self.median_): self.median_ = 1000000
        return self
    def transform(self, X):
        X_new = X.copy()
        X_new['is_budget_missing'] = X_new['budget_parsed'].isna().astype(int)
        temp_budget = X_new['budget_parsed'].fillna(self.median_)
        X_new['budget_log'] = np.log1p(temp_budget)
        return X_new.drop(columns=['budget_parsed'], errors='ignore')

class AdvancedCastEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, min_count=5, smoothing_m=30):
        self.min_count = min_count; self.smoothing_m = smoothing_m
    def fit(self, X, y):
        df_temp = pd.DataFrame({'actors': X['lead_actors_ids'], 'rating': y})
        def safe_split(val): return [a.strip() for a in str(val).strip().split(',') if a.strip() not in ['', 'nan', 'None']] if pd.notna(val) else []
        df_temp['actors'] = df_temp['actors'].apply(safe_split)
        df_exploded = df_temp.explode('actors').dropna(subset=['actors'])
        self.global_mean_ = y.mean()
        stats = df_exploded.groupby('actors')['rating'].agg(['mean', 'count'])
        stats = stats[stats['count'] >= self.min_count]
        stats['smoothed'] = ((stats['count'] * stats['mean']) + (self.smoothing_m * self.global_mean_)) / (stats['count'] + self.smoothing_m)
        self.actor_stats_ = stats[['smoothed', 'count']].to_dict('index')
        return self
    def transform(self, X):
        X_new = X.copy()
        def calculate_cast_stats(actors_str):
            if pd.isna(actors_str): return pd.Series({'cast_mean': self.global_mean_, 'cast_max': self.global_mean_, 'cast_min': self.global_mean_, 'cast_experience_log': 0})
            actors = [a.strip() for a in str(actors_str).strip().split(',') if a.strip() not in ['', 'nan', 'None']]
            if not actors: return pd.Series({'cast_mean': self.global_mean_, 'cast_max': self.global_mean_, 'cast_min': self.global_mean_, 'cast_experience_log': 0})
            scores = [self.actor_stats_.get(a, {'smoothed': self.global_mean_})['smoothed'] for a in actors]
            counts = [self.actor_stats_.get(a, {'count': 0})['count'] for a in actors]
            return pd.Series({'cast_mean': np.mean(scores), 'cast_max': np.max(scores), 'cast_min': np.min(scores), 'cast_experience_log': np.log1p(np.sum(counts))})
        stats_df = X_new['lead_actors_ids'].apply(calculate_cast_stats)
        X_new = pd.concat([X_new, stats_df], axis=1)
        return X_new.drop(columns=['lead_actors_ids'], errors='ignore')

In [4]:
# ==============================================================================
# תא 4: טעינת נתונים 
# ==============================================================================
print("Loading dataset...")
raw_df = pd.read_csv(DATASET_PATH, low_memory=False)

print("Running prepare_data...")
df_processed = prepare_data(raw_df)

df_model = df_processed.dropna(subset=['averageRating']).copy()
y = df_model['averageRating']
X = df_model.drop(columns=['averageRating'])

print(f"Data ready. X shape: {X.shape}")

Loading dataset...
Running prepare_data...
Data ready. X shape: (115560, 41)


In [6]:
# ==============================================================================
# תא 4.5: כיוונון היפר-פרמטרים (Hyperparameter Tuning) למודל הלינארי
# ==============================================================================
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
import time

print("🔍 מבצע חיפוש Grid Search למציאת הפרמטרים האידיאליים עבור Elastic Net...")

# בונים פייפליין זהה, אבל משאירים את ה-ElasticNet "ריק" מפרמטרים
tune_pipeline = Pipeline([
    ('genre_encoder', GenreTargetEncoder(min_count=10, smoothing_m=20)),
    ('cast_encoder', AdvancedCastEncoder(min_count=5, smoothing_m=30)), 
    ('runtime_encoder', RuntimeHighPolynomial()), 
    ('budget_encoder', BudgetLogWithMissingFlag()), 
    ('imputer', SimpleImputer(strategy='median')), 
    ('scaler', StandardScaler()), 
    ('elasticnet', ElasticNet(random_state=42, max_iter=2000)) 
])

# מגדירים את רשת החיפוש )
param_grid = {
    'elasticnet__alpha': [0.001, 0.01, 0.1, 1.0],
    'elasticnet__l1_ratio': [0.01,0.1,0.25, 0.5, 0.7, 0.9]
}

start_tune = time.time()

# מריצים את החיפוש עם 10 קיפולים (כדי שירוץ יחסית מהר ולא יתקע לך את המחשב)
grid_search = GridSearchCV(
    estimator=tune_pipeline, 
    param_grid=param_grid, 
    cv=10, 
    scoring='neg_root_mean_squared_error', 
    n_jobs=-1
)

grid_search.fit(X, y)
tune_time = time.time() - start_tune

print(f"⏱️ זמן חיפוש: {tune_time:.1f} שניות")
print(f"✅ הפרמטרים המנצחים שנבחרו סטטיסטית הם: {grid_search.best_params_}")

🔍 מבצע חיפוש Grid Search למציאת הפרמטרים האידיאליים עבור Elastic Net...
⏱️ זמן חיפוש: 2673.8 שניות
✅ הפרמטרים המנצחים שנבחרו סטטיסטית הם: {'elasticnet__alpha': 0.01, 'elasticnet__l1_ratio': 0.05}


In [9]:
# ==============================================================================
# תא 5: יצירת ה-Pipeline, הרצת 10-Fold CV והדפסת התוצאות לכל פולד ולסיכום
# ==============================================================================
pipeline = Pipeline([
    ('genre_encoder', GenreTargetEncoder(min_count=10, smoothing_m=20)),
    ('cast_encoder', AdvancedCastEncoder(min_count=5, smoothing_m=30)), 
    ('runtime_encoder', RuntimeHighPolynomial()), 
    ('budget_encoder', BudgetLogWithMissingFlag()), 
    ('imputer', SimpleImputer(strategy='median')), 
    ('scaler', StandardScaler()), 
    ('elasticnet', ElasticNet(alpha=0.01, l1_ratio=0.01, random_state=42, max_iter=2000)) 
])

print("Running 10-Fold Cross Validation...")
cv_strategy = KFold(n_splits=10, shuffle=True, random_state=42)
scoring_metrics = {
    'R2': 'r2', 
    'RMSE': 'neg_root_mean_squared_error', 
    'MAE': 'neg_mean_absolute_error'
}

start_time = time.time()
cv_results = cross_validate(
    estimator=pipeline, 
    X=X, 
    y=y, 
    cv=cv_strategy, 
    scoring=scoring_metrics, 
    n_jobs=-1
)
run_time = time.time() - start_time

r2_scores = cv_results['test_R2']
rmse_scores = -cv_results['test_RMSE']
mae_scores = -cv_results['test_MAE']

print(f"Execution time: {run_time:.1f} seconds\n")
print("Detailed metrics per fold:")
print("-" * 50)
for i in range(10):
    print(f"Fold {i+1:02d} | R²: {r2_scores[i]:.4f} | RMSE: {rmse_scores[i]:.4f} | MAE: {mae_scores[i]:.4f}")

print("\nFinal Performance Summary (Mean ± Std):")
print("-" * 50)
print(f"R²   : {np.mean(r2_scores):.4f} (± {np.std(r2_scores):.4f})")
print(f"RMSE : {np.mean(rmse_scores):.4f} (± {np.std(rmse_scores):.4f})")
print(f"MAE  : {np.mean(mae_scores):.4f} (± {np.std(mae_scores):.4f})")

Running 10-Fold Cross Validation...
Execution time: 57.7 seconds

Detailed metrics per fold:
--------------------------------------------------
Fold 01 | R²: 0.2667 | RMSE: 1.1111 | MAE: 0.8514
Fold 02 | R²: 0.2617 | RMSE: 1.0962 | MAE: 0.8334
Fold 03 | R²: 0.2617 | RMSE: 1.1108 | MAE: 0.8475
Fold 04 | R²: 0.2571 | RMSE: 1.1235 | MAE: 0.8569
Fold 05 | R²: 0.2661 | RMSE: 1.1052 | MAE: 0.8433
Fold 06 | R²: 0.2546 | RMSE: 1.1214 | MAE: 0.8482
Fold 07 | R²: 0.2574 | RMSE: 1.1211 | MAE: 0.8526
Fold 08 | R²: 0.2761 | RMSE: 1.0999 | MAE: 0.8417
Fold 09 | R²: 0.2675 | RMSE: 1.1068 | MAE: 0.8407
Fold 10 | R²: 0.2740 | RMSE: 1.0919 | MAE: 0.8374

Final Performance Summary (Mean ± Std):
--------------------------------------------------
R²   : 0.2643 (± 0.0068)
RMSE : 1.1088 (± 0.0104)
MAE  : 0.8453 (± 0.0069)


In [ ]:
# ==============================================================================
# תא 6: אימון על כל הסט ושמירת המודל ל-Pickle
# ==============================================================================
print("Fitting the pipeline on the full dataset...")
pipeline.fit(X, y)

print(f"Saving the model to {MODEL_PICKLE_PATH}...")
with open(MODEL_PICKLE_PATH, 'wb') as f:
    pickle.dump(pipeline, f)
    
print("Model saved successfully.")

In [ ]:
# ==============================================================================
# תא 7.5: כיוונון היפר-פרמטרים למודל ה-Random Forest (Grid Search)
# ==============================================================================
from sklearn.model_selection import GridSearchCV
import time
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
import time

print("🔍 מבצע חיפוש Grid Search למציאת הפרמטרים האידיאליים עבור Random Forest...")

# בניית פייפליין לחיפוש (ללא הפרמטרים הספציפיים בעץ)
rf_tune_pipeline = Pipeline([
    ('genre_encoder', GenreTargetEncoder(min_count=10, smoothing_m=20)),
    ('cast_encoder', AdvancedCastEncoder(min_count=5, smoothing_m=30)), 
    ('runtime_encoder', RuntimeHighPolynomial()), 
    ('budget_encoder', BudgetLogWithMissingFlag()), 
    ('imputer', SimpleImputer(strategy='median')), 
    ('scaler', StandardScaler()), 
    ('rf', RandomForestRegressor(random_state=42, n_jobs=-1)) # n_jobs=-1 כדי שירוץ מהר על כל הליבות
])

# הגדרת רשת החיפוש (כוללת את הערכים שלך ועוד כמה אופציות לבדיקה)
rf_param_grid = {
    'rf__n_estimators': [50, 100, 150],       # כמות העצים ביער
    'rf__max_depth': [10, 15, 20],            # עומק מקסימלי של כל עץ
    'rf__min_samples_split': [5, 10],         # מינימום דוגמאות לפיצול צומת
    'rf__min_samples_leaf': [2, 5]            # מינימום דוגמאות בעלה סופי
}

start_rf_tune = time.time()

# מריצים את החיפוש עם 3 קיפולים (כדי לחסוך זמן ריצה על מודל כבד כזה)
rf_grid_search = GridSearchCV(
    estimator=rf_tune_pipeline, 
    param_grid=rf_param_grid, 
    cv=10, 
    scoring='neg_root_mean_squared_error', 
    n_jobs=-1
)

rf_grid_search.fit(X, y)
rf_tune_time = time.time() - start_rf_tune

print(f"⏱️ זמן חיפוש: {rf_tune_time:.1f} שניות")
print(f"✅ הפרמטרים המנצחים שנבחרו סטטיסטית הם:\n{rf_grid_search.best_params_}")

🔍 מבצע חיפוש Grid Search למציאת הפרמטרים האידיאליים עבור Random Forest...


In [ ]:
# ==============================================================================
# תא 8: מודל שני לבחירה - Random Forest (הרצת 10-Fold CV)
# ==============================================================================
from sklearn.ensemble import RandomForestRegressor

print("🛠️ בונה Pipeline עבור Random Forest...")
# שימוש במחלקות מניעת הזליגה שכבר הגדרנו בתאים קודמים
rf_pipeline = Pipeline([
    ('genre_encoder', GenreTargetEncoder(min_count=10, smoothing_m=20)),
    ('cast_encoder', AdvancedCastEncoder(min_count=5, smoothing_m=30)), 
    ('runtime_encoder', RuntimeHighPolynomial()), # RF לא חייב פולינומים, אבל זה לא מזיק ונשאר אחיד
    ('budget_encoder', BudgetLogWithMissingFlag()), 
    ('imputer', SimpleImputer(strategy='median')), 
    ('scaler', StandardScaler()), 
    ('rf', RandomForestRegressor(
        n_estimators=100,        # 100 עצים
        max_depth=15,            # הגבלת עומק כדי לא לעשות Overfitting (קריטי ב-RF)
        min_samples_split=10, 
        min_samples_leaf=5, 
        random_state=42, 
        n_jobs=-1                # שימוש בכל הליבות לחישוב מהיר
    )) 
])

print("📊 מריץ 10-Fold Cross Validation עבור Random Forest (ייקח קצת זמן)...")
rf_cv_results = cross_validate(
    estimator=rf_pipeline, 
    X=X,  # ה-X וה-y כבר מוגדרים מהרצת הנתונים בתא 4
    y=y, 
    cv=cv_strategy, # ה-KFold מוגדר מתא 5
    scoring=scoring_metrics, 
    n_jobs=-1
)

# חילוץ המדדים
rf_r2_scores = rf_cv_results['test_R2']
rf_rmse_scores = -rf_cv_results['test_RMSE']
rf_mae_scores = -rf_cv_results['test_MAE']

print("===========================================================")
print(" 🌳 מדדי ביצוע מפורטים לכל Fold (Random Forest)")
print("===========================================================")
for i in range(10):
    print(f"Fold {i+1:02d} -> R²: {rf_r2_scores[i]:.4f} | RMSE: {rf_rmse_scores[i]:.4f} | MAE: {rf_mae_scores[i]:.4f}")

print("===========================================================")
print(" 🏆 סיכום ביצועים - Random Forest (ממוצע וסטיית תקן)")
print("===========================================================")
print(f" R²   : {np.mean(rf_r2_scores):.4f}  (± {np.std(rf_r2_scores):.4f})")
print(f" RMSE : {np.mean(rf_rmse_scores):.4f}  (± {np.std(rf_rmse_scores):.4f})")
print(f" MAE  : {np.mean(rf_mae_scores):.4f}  (± {np.std(rf_mae_scores):.4f})")
print("===========================================================")

In [ ]:
# ==============================================================================
# תא 9: אימון סופי על 100% מהנתונים ושמירת המודל (Random Forest)
# ==============================================================================
RF_MODEL_PICKLE_PATH = 'rf_movie_rating_pipeline.pkl'

print("💾 מאמן את מודל ה-Random Forest על כל הנתונים...")
rf_pipeline.fit(X, y)

print(f"💾 שומר את מודל ה-Random Forest לקובץ: {RF_MODEL_PICKLE_PATH}")
with open(RF_MODEL_PICKLE_PATH, 'wb') as f:
    pickle.dump(rf_pipeline, f)
    
print("✅ השמירה הושלמה! יש לך עכשיו שני מודלים מוכנים.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# הגדרת סגנון הגרף
plt.style.use('seaborn-v0_8-whitegrid')
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# שליפת נתוני התקציב שעברו ניקוי (ללא ערכים חסרים)
clean_budgets = df_processed['budget_parsed'].dropna()

# יצירת משתנה הלוגריתם
log_budgets = np.log1p(clean_budgets)

# גרף 1: התפלגות מקורית (מוטה מאוד ימינה)
sns.histplot(clean_budgets, bins=50, ax=axes[0], color='salmon', kde=True)
axes[0].set_title('Raw Budget Distribution (After Inflation & Currency)', fontsize=14)
axes[0].set_xlabel('Budget (USD)', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].ticklabel_format(style='sci', axis='x', scilimits=(0,0)) # תצוגה מדעית למספרים גדולים

# גרף 2: התפלגות אחרי טרנספורמציית הלוגריתם (דומה לנורמלית)
sns.histplot(log_budgets, bins=50, ax=axes[1], color='skyblue', kde=True)
axes[1].set_title('Log Transformed Budget Distribution (budget_log)', fontsize=14)
axes[1].set_xlabel('Log(1 + Budget)', fontsize=12)
axes[1].set_ylabel('Count', fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# תא 10: ניתוח שגיאות (Error Analysis) - חילוץ Over/Under Predictions
# ==============================================================================
from sklearn.model_selection import cross_val_predict

print("🔍 מחלץ תחזיות עבור כל סרט באמצעות Cross-Validation...")
# נשתמש במודל ה-Random Forest (rf_pipeline) בהנחה שהוא הטוב יותר. 
# אם ה-ElasticNet ניצח, תשנה כאן ל-elastic_pipeline
predictions = cross_val_predict(rf_pipeline, X, y, cv=cv_strategy, n_jobs=-1)

# יצירת דאטה-פריים להשוואה בין התחזית לאמת
error_df = pd.DataFrame({
    'tconst': raw_df.loc[X.index, 'tconst'], # מזהה הסרט כדי שנוכל לחפש את השם שלו
    'primaryTitle': raw_df.loc[X.index, 'primaryTitle'], # שם הסרט המקורי
    'startYear': raw_df.loc[X.index, 'startYear'],
    'Actual_Rating': y,
    'Predicted_Rating': predictions
})

# חישוב השגיאה (Residuals)
# שגיאה חיובית = המודל חזה יותר מדי (Overprediction)
# שגיאה שלילית = המודל חזה מעט מדי (Underprediction)
error_df['Error'] = error_df['Predicted_Rating'] - error_df['Actual_Rating']

# מציאת 10 הסרטים עם ה-Overprediction הגדול ביותר
# אלו סרטים שהמודל חשב שהם "שוברי קופות מצוינים" (לפי השחקנים/תקציב) אבל בפועל הם היו גרועים
top_10_over = error_df.sort_values(by='Error', ascending=False).head(10)

# מציאת 10 הסרטים עם ה-Underprediction הגדול ביותר
# אלו סרטים "פנינים נסתרות" שהמודל זלזל בהם (אולי כי הם ישנים או ללא קאסט מוכר), אבל הקהל אהב
top_10_under = error_df.sort_values(by='Error', ascending=True).head(10)

# הדפסה יפה לטובת הדוח
print("\n" + "="*80)
print(" 🛑 TOP 10 OVER-PREDICTIONS (המודל נתן ציון גבוה מדי)")
print("="*80)
display(top_10_over[['primaryTitle', 'startYear', 'Actual_Rating', 'Predicted_Rating', 'Error']].round(2))

print("\n" + "="*80)
print(" 💎 TOP 10 UNDER-PREDICTIONS (המודל נתן ציון נמוך מדי)")
print("="*80)
display(top_10_under[['primaryTitle', 'startYear', 'Actual_Rating', 'Predicted_Rating', 'Error']].round(2))

In [ ]:
# רשימת 5 הסרטים שבחרנו לניתוח
selected_movies = [
    'Justin Bieber: Never Say Never',
    'Tribalism Is Killing Us',
    'Bukunja Tekunja Mitti: The Cannibals',
    'Halloween Deluxe',
    'Justicia Implacable'
]

print("שולף נתונים טכניים לניתוח שגיאות:\n" + "="*50)

for movie in selected_movies:
    # חיפוש הסרט בדאטה הגולמי
    movie_data = raw_df[raw_df['primaryTitle'] == movie]
    
    if not movie_data.empty:
        row = movie_data.iloc[0]
        print(f"סרט: {movie}")
        print(f"ציון בפועל: {row.get('averageRating', 'N/A')} | כמות הצבעות (numVotes): {row.get('numVotes', 'N/A')}")
        print(f"תקציב גולמי: {row.get('budget', 'N/A')}")
        print(f"ז'אנרים: {row.get('genres', 'N/A')}")
        print(f"מדינה: {row.get('Country', 'N/A')} | שפה: {row.get('Language', 'N/A')}")
        print(f"קאסט (IDs): {str(row.get('lead_actors_ids', 'N/A'))[:60]}...")
        print("-" * 50)

In [ ]:
# ==============================================================================
# תא 11: בניית טבלת השוואה ומציאת חריגים מוחלטים (Top 20 Outliers)
# ==============================================================================
import pandas as pd
from sklearn.model_selection import cross_val_predict

print("⏳ מחשב תחזיות לשני המודלים ומייצר טבלת השוואה...")

# 1. יצירת התחזיות באמצעות Cross-Validation
rf_preds = cross_val_predict(rf_pipeline, X, y, cv=cv_strategy, n_jobs=-1)
en_preds = cross_val_predict(pipeline, X, y, cv=cv_strategy, n_jobs=-1)

# 2. בניית טבלת ההשוואה (מונע את שגיאת ה-NameError)
df_comp = pd.DataFrame({
    'Title': raw_df.loc[X.index, 'primaryTitle'],
    'Actual': y,
    'RF_Pred': rf_preds,
    'EN_Pred': en_preds
})

# 3. חישוב השגיאות (רגילות ומוחלטות)
df_comp['RF_Error'] = df_comp['Actual'] - df_comp['RF_Pred']
df_comp['EN_Error'] = df_comp['Actual'] - df_comp['EN_Pred']

df_comp['RF_Abs_Error'] = df_comp['RF_Error'].abs()
df_comp['EN_Abs_Error'] = df_comp['EN_Error'].abs()

# 4. חילוץ 20 החריגים עם השגיאה המוחלטת הגדולה ביותר (הפספוסים הגדולים ביותר)
rf_top_20 = df_comp.sort_values(by='RF_Abs_Error', ascending=False).head(20)
en_top_20 = df_comp.sort_values(by='EN_Abs_Error', ascending=False).head(20)

# 5. חיתוך ובדיקת חפיפות באמצעות קבוצות (Sets)
rf_titles = set(rf_top_20['Title'])
en_titles = set(en_top_20['Title'])

overlapping_movies = rf_titles.intersection(en_titles)
non_overlapping_movies = (rf_titles - en_titles).union(en_titles - rf_titles)

# 6. הדפסה נקייה וממוקדת לדוח
print("==================================================")
print(" 🎬 השוואת חריגים (Top 20 Outliers)")
print("==================================================")
print(f"✅ כמות סרטים חופפים (טעות זהה בשני המודלים): {len(overlapping_movies)}")
print(f"❌ כמות סרטים לא חופפים (טעות ייחודית למודל): {len(non_overlapping_movies)}")
print("-" * 50)
print("📋 רשימת הסרטים החופפים (סרטים ש'שברו' את שני האלגוריתמים):")
for i, movie in enumerate(sorted(list(overlapping_movies)), 1):
    print(f"   {i}. {movie}")

In [ ]:
# ==============================================================================
# תא 11: השוואת חריגים (Outliers) בין שני המודלים (סעיף 5.3)
# ==============================================================================

# חישוב שגיאה מוחלטת כדי למצוא את 20 הטעויות הגדולות ביותר (ללא תלות בכיוון השגיאה)
df_comp['RF_Abs_Error'] = df_comp['RF_Error'].abs()
df_comp['EN_Abs_Error'] = df_comp['EN_Error'].abs()

# שליפת 20 הסרטים עם השגיאה המוחלטת הגדולה ביותר מכל מודל
rf_top_20 = df_comp.sort_values(by='RF_Abs_Error', ascending=False).head(20)
en_top_20 = df_comp.sort_values(by='EN_Abs_Error', ascending=False).head(20)

# המרה לקבוצות (Sets) של שמות סרטים לצורך פעולות חיתוך והשוואה
rf_titles = set(rf_top_20['Title'])
en_titles = set(en_top_20['Title'])

# מציאת הסרטים החופפים והלא-חופפים
overlapping_movies = rf_titles.intersection(en_titles)
non_overlapping_movies = (rf_titles - en_titles).union(en_titles - rf_titles)

# הדפסה נקייה וממוקדת לדוח
print("==================================================")
print(" 🎬 השוואת חריגים (Top 20 Outliers)")
print("==================================================")
print(f"✅ כמות סרטים חופפים (טעות זהה בשני המודלים): {len(overlapping_movies)}")
print(f"❌ כמות סרטים לא חופפים (טעות ייחודית למודל): {len(non_overlapping_movies)}")
print("-" * 50)
print("📋 רשימת הסרטים החופפים:")
for i, movie in enumerate(sorted(list(overlapping_movies)), 1):
    print(f"   {i}. {movie}")


In [ ]:
# ==============================================================================
# תא 12 (מתוקן וחסין תקלות): מבחן Levene למובהקות סטטיסטית על הנתונים המעובדים
# ==============================================================================
import scipy.stats as stats
import pandas as pd
import numpy as np

print("📊 מחשב מובהקות סטטיסטית (P-Values) עבור המאפיינים (אחרי עיבוד)...")

# 1. הגדרת מי ניצח על בסיס הטבלה הקיים
conditions = [
    (df_comp['EN_Abs_Error'] > df_comp['RF_Abs_Error'] + 0.5), # RF ניצח
    (df_comp['RF_Abs_Error'] > df_comp['EN_Abs_Error'] + 0.5)  # EN ניצח
]
choices = ['RF_Won', 'EN_Won']
df_comp['Winner'] = np.select(conditions, choices, default='Tie')

# 2. העברת הנתונים דרך שלב העיבוד של הפייפליין
X_transformed = pipeline[:-1].transform(X)

# 3. חילוץ שמות חסין תקלות (פותר את שגיאת ה-Shape המקורית)
try:
    feature_names = list(pipeline[:-1].get_feature_names_out())
except:
    # גיבוי חכם: לוקח את השמות המקוריים ומוסיף שמות גנריים רק לעמודות החדשות שנוצרו
    orig_cols = list(X.columns)
    num_features = X_transformed.shape[1]
    num_orig = len(orig_cols)
    
    if num_features == num_orig:
        feature_names = orig_cols
    else:
        extra_cols = [f"Engineered_Feature_{i}" for i in range(num_orig, num_features)]
        feature_names = orig_cols + extra_cols

# בניית הדאטה-פריים המעובד עם כמות עמודות תואמת לחלוטין
df_features = pd.DataFrame(X_transformed, columns=feature_names, index=X.index)
df_features['Winner'] = df_comp['Winner']

# סינון רק לסרטים בהם יש מנצח ברור (ללא תוצאות תיקו)
df_winners = df_features[df_features['Winner'].isin(['RF_Won', 'EN_Won'])]

# 4. ריצה על כל הפיצ'רים וחישוב מבחן Levene לשוויון שונויות
results = []
for col in feature_names:
    rf_data = df_winners[df_winners['Winner'] == 'RF_Won'][col]
    en_data = df_winners[df_winners['Winner'] == 'EN_Won'][col]
    
    # חישוב רק אם יש שונות מינימלית בשתי הקבוצות (מונע שגיאות NaN)
    if len(rf_data) > 1 and len(en_data) > 1 and rf_data.std() > 0.001 and en_data.std() > 0.001:
        stat, p_val = stats.levene(rf_data, en_data, center='median')
        results.append({
            'Feature': col,
            'Std_RF_Won': rf_data.std(),
            'Std_EN_Won': en_data.std(),
            'P_Value': p_val
        })

# 5. יצירת טבלת תוצאות ומיון מהפיצ'ר הכי מובהק ומטה
p_val_df = pd.DataFrame(results).sort_values(by='P_Value')

print("\n✅ הטבלה מוכנה! המאפיינים בעלי ההבדל הסטטיסטי המובהק ביותר:")
display(p_val_df.head(10).round(4))

print("\n💡 מקרא לדוח:")
print("- פיצ'ר מוגדר כמובהק אם P_Value < 0.05 (השונות בין קבוצות הניצחון שונה בוודאות).")

In [ ]:
# ==============================================================================# ==============================================================================
# תא 12: מבחן Levene למובהקות סטטיסטית - עבור המאפיינים הספציפיים לדוח
# ==============================================================================
import scipy.stats as stats
import pandas as pd
import numpy as np

print("📊 מחשב מובהקות סטטיסטית עבור: מדינה, שפה ומידע חסר (לאחר עיבוד)...")

# 1. הגדרת המנצח (RF מול EN)
conditions = [
    (df_comp['EN_Abs_Error'] > df_comp['RF_Abs_Error'] + 0.5), # RF ניצח
    (df_comp['RF_Abs_Error'] > df_comp['EN_Abs_Error'] + 0.5)  # EN ניצח
]
choices = ['RF_Won', 'EN_Won']
df_comp['Winner'] = np.select(conditions, choices, default='Tie')

# 2. העברת הנתונים דרך הפייפליין וחילוץ שמות בטוח
X_transformed = pipeline[:-1].transform(X)

try:
    feature_names = list(pipeline[:-1].get_feature_names_out())
except:
    orig_cols = list(X.columns)
    num_features = X_transformed.shape[1]
    num_orig = len(orig_cols)
    if num_features == num_orig:
        feature_names = orig_cols
    else:
        extra_cols = [f"Engineered_Feature_{i}" for i in range(num_orig, num_features)]
        feature_names = orig_cols + extra_cols

df_features = pd.DataFrame(X_transformed, columns=feature_names, index=X.index)
df_features['Winner'] = df_comp['Winner']

# סינון רק לסרטים בהם יש מנצח ברור
df_winners = df_features[df_features['Winner'].isin(['RF_Won', 'EN_Won'])]

# 3. רשימת המאפיינים הספציפית שביקשת לדוח!
selected_features = ['is_other_country', 'is_lang_Spanish', 'is_decade_unknown']

results = []
for col in selected_features:
    # מנגנון חכם למציאת העמודה, גם אם הפייפליין הוסיף לה קידומת לשם
    matching_cols = [c for c in df_winners.columns if col in c]
    
    if len(matching_cols) > 0:
        actual_col_name = matching_cols[0] # ניקח את ההתאמה המדויקת מהדאטה
        rf_data = df_winners[df_winners['Winner'] == 'RF_Won'][actual_col_name]
        en_data = df_winners[df_winners['Winner'] == 'EN_Won'][actual_col_name]
        
        # חישוב המבחן רק אם יש שונות (למניעת שגיאת NaN)
        if rf_data.std() > 0.001 and en_data.std() > 0.001:
            stat, p_val = stats.levene(rf_data, en_data, center='median')
        else:
            p_val = np.nan # יופיע רק אם הנתונים היו 100% אפסים בשתי הקבוצות
            
        results.append({
            'Feature': col, # מציג את השם הנקי שביקשת
            'Std_RF_Won': rf_data.std(),
            'Std_EN_Won': en_data.std(),
            'P_Value': p_val
        })
    else:
        print(f"⚠️ אזהרה: העמודה {col} לא נמצאה לאחר העיבוד.")

# 4. הדפסת הטבלה שרצית
final_p_val_df = pd.DataFrame(results)
display(final_p_val_df.round(4))

In [ ]:
# ==============================================================================
# תא 14: Fairness Analysis (מודל Random Forest) - קוד מותאם אקדמית/תעשייתית
# ==============================================================================
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import cross_val_predict

print("⏳ מכין את טבלת הנתונים לניתוח הוגנות...")

# ---------------------------------------------------------
# 0. בניית טבלת df_fairness מתוך הנתונים המקוריים והתחזיות
# ---------------------------------------------------------
df_fairness = raw_df.loc[X.index, ['genres', 'Country', 'startYear', 'budget']].copy()
df_fairness['Actual'] = y

# בדיקה: אם התחזיות כבר חושבו קודם, נשתמש בהן כדי לחסוך זמן. אם לא, נחשב עכשיו.
if 'rf_preds' in locals():
    df_fairness['RF_Pred'] = rf_preds
else:
    df_fairness['RF_Pred'] = cross_val_predict(rf_pipeline, X, y, cv=cv_strategy, n_jobs=-1)


# פונקציית עזר לחישוב גם RMSE וגם MAE
def calc_metrics(actual, pred):
    rmse = np.sqrt(mean_squared_error(actual, pred))
    mae = mean_absolute_error(actual, pred)
    return rmse, mae

print("================================================================")
print(" ⚖️ סעיף 6: Fairness Analysis (מודל Random Forest) - גרסה מתקדמת")
print("================================================================")

# ---------------------------------------------------------
# 1. טיפול מקצועי בז'אנרים (Multi-Label Parsing & Exploding)
# ---------------------------------------------------------
print("\n1. ביצועים לפי 5 הז'אנרים הנפוצים ביותר:")

# פונקציה חכמה לחילוץ ז'אנרים ממחרוזות מבולגנות לרשימות נקיות
def extract_genres_to_list(val):
    if pd.isna(val) or str(val).strip() == '':
        return []
    # מחיקת סוגריים ומרכאות, ופיצול חכם לפי פסיק
    clean_str = str(val).replace('[', '').replace(']', '').replace("'", "").replace('"', '')
    return [g.strip() for g in clean_str.split(',') if g.strip()]

# יצירת עמודת רשימות (Lists)
df_fairness['genres_list'] = df_fairness['genres'].apply(extract_genres_to_list)

# "פיצוץ" העמודה לשורות נפרדות עבור כל ז'אנר (הסטנדרט לניתוח Multi-label)
df_exploded = df_fairness.explode('genres_list')
df_exploded = df_exploded.dropna(subset=['genres_list'])

# מציאת 5 הז'אנרים הנפוצים ביותר
top_5_genres = df_exploded['genres_list'].value_counts().head(5).index.tolist()

for genre in top_5_genres:
    # סינון מתוך המאגר המפוצץ כדי לקבל בדיוק את כל הסרטים שמכילים את הז'אנר
    subset = df_exploded[df_exploded['genres_list'] == genre]
    if len(subset) > 0:
        rmse_val, mae_val = calc_metrics(subset['Actual'], subset['RF_Pred'])
        print(f"   - {genre:<12} (n={len(subset):,}): RMSE = {rmse_val:.3f} | MAE = {mae_val:.3f}")

# ---------------------------------------------------------
# 2. לפי מדינת מקור (US מול Non-US)
# ---------------------------------------------------------
print("\n2. ביצועים לפי מדינת מקור (US מול Non-US):")
mask_us = df_fairness['Country'].str.contains('United States', na=False)
subset_us = df_fairness[mask_us]
subset_non_us = df_fairness[~mask_us]

if len(subset_us) > 0:
    rmse_val, mae_val = calc_metrics(subset_us['Actual'], subset_us['RF_Pred'])
    print(f"   - ארצות הברית (n={len(subset_us):,}): RMSE = {rmse_val:.3f} | MAE = {mae_val:.3f}")

if len(subset_non_us) > 0:
    rmse_val, mae_val = calc_metrics(subset_non_us['Actual'], subset_non_us['RF_Pred'])
    print(f"   - מחוץ לארה\"ב (n={len(subset_non_us):,}): RMSE = {rmse_val:.3f} | MAE = {mae_val:.3f}")

# ---------------------------------------------------------
# 3. לפי עשור יציאה (שנות ה-90, ה-2000 וה-2010)
# ---------------------------------------------------------
print("\n3. ביצועים לפי עשור יציאה (1990, 2000, 2010):")
df_fairness['startYear'] = pd.to_numeric(df_fairness['startYear'], errors='coerce')
df_fairness['Decade'] = (df_fairness['startYear'] // 10) * 10
target_decades = [1990, 2000, 2010]

for decade in target_decades:
    subset = df_fairness[df_fairness['Decade'] == decade]
    if len(subset) > 0:
        rmse_val, mae_val = calc_metrics(subset['Actual'], subset['RF_Pred'])
        print(f"   - שנות ה-{decade} (n={len(subset):,}): RMSE = {rmse_val:.3f} | MAE = {mae_val:.3f}")

# ---------------------------------------------------------
# 4. לפי תקציב (שלישונים) - רק לסרטים עם תקציב ידוע
# ---------------------------------------------------------
print("\n4. ביצועים לפי תקציב (שלישונים - נמוך/בינוני/גבוה):")
df_fairness['budget_clean'] = pd.to_numeric(df_fairness['budget'].astype(str).str.replace(r'[^\d.]', '', regex=True), errors='coerce')
with_budget = df_fairness[df_fairness['budget_clean'].notna()].copy()

if len(with_budget) > 0:
    with_budget['Budget_Tertile'] = pd.qcut(with_budget['budget_clean'], q=3, labels=['נמוך (Low)', 'בינוני (Medium)', 'גבוה (High)'])
    for tertile in ['נמוך (Low)', 'בינוני (Medium)', 'גבוה (High)']:
        subset = with_budget[with_budget['Budget_Tertile'] == tertile]
        if len(subset) > 0:
            rmse_val, mae_val = calc_metrics(subset['Actual'], subset['RF_Pred'])
            print(f"   - תקציב {tertile} (n={len(subset):,}): RMSE = {rmse_val:.3f} | MAE = {mae_val:.3f}")

# ---------------------------------------------------------
# 5. חסר נתונים מול מלא
# ---------------------------------------------------------
print("\n5. ביצועים לפי שלמות נתונים (תקציב חסר מול קיים):")
missing_budget = df_fairness[df_fairness['budget_clean'].isna()]

if len(with_budget) > 0:
    rmse_val, mae_val = calc_metrics(with_budget['Actual'], with_budget['RF_Pred'])
    print(f"   - תקציב תקין וידוע (n={len(with_budget):,}): RMSE = {rmse_val:.3f} | MAE = {mae_val:.3f}")

if len(missing_budget) > 0:
    rmse_val, mae_val = calc_metrics(missing_budget['Actual'], missing_budget['RF_Pred'])
    print(f"   - תקציב חסר (n={len(missing_budget):,}): RMSE = {rmse_val:.3f} | MAE = {mae_val:.3f}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# ==========================================
# 1. הגדרת הנתונים
# ==========================================

# נתוני ז'אנרים (ממוינים מהביצועים הטובים ביותר לגרועים ביותר)
genres_data = {
    'Category': ['Romance', 'Documentary', 'Drama', 'Comedy', 'Action'],
    'RMSE': [0.998, 1.009, 1.032, 1.103, 1.165],
    'MAE': [0.760, 0.740, 0.783, 0.840, 0.895]
}

# נתוני עשורים (ממוינים כרונולוגית)
decades_data = {
    'Category': ['1990s', '2000s', '2010s'],
    'RMSE': [1.030, 1.121, 1.159],
    'MAE': [0.796, 0.850, 0.875]
}

df_genres = pd.DataFrame(genres_data)
df_decades = pd.DataFrame(decades_data)

# ==========================================
# 2. עיצוב ויצירת הגרפים
# ==========================================

plt.style.use('seaborn-v0_8-whitegrid')
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

width = 0.35 # עובי כל עמודה

# --- גרף 1: ז'אנרים ---
x1 = np.arange(len(df_genres['Category']))
axes[0].bar(x1 - width/2, df_genres['RMSE'], width, label='RMSE', color='#4C72B0')
axes[0].bar(x1 + width/2, df_genres['MAE'], width, label='MAE', color='#DD8452')

axes[0].set_title('Model Errors by Top 5 Genres', fontsize=14, pad=15, fontweight='bold')
axes[0].set_ylabel('Error Score', fontsize=12)
axes[0].set_xticks(x1)
axes[0].set_xticklabels(df_genres['Category'], fontsize=11)
axes[0].legend()
axes[0].set_ylim(0, 1.4)

# הוספת תוויות נתונים מעל העמודות (ז'אנרים)
for i in range(len(x1)):
    axes[0].text(x1[i] - width/2, df_genres['RMSE'][i] + 0.02, f"{df_genres['RMSE'][i]:.3f}", ha='center', va='bottom', fontsize=10)
    axes[0].text(x1[i] + width/2, df_genres['MAE'][i] + 0.02, f"{df_genres['MAE'][i]:.3f}", ha='center', va='bottom', fontsize=10)


# --- גרף 2: עשורים ---
x2 = np.arange(len(df_decades['Category']))
axes[1].bar(x2 - width/2, df_decades['RMSE'], width, label='RMSE', color='#4C72B0')
axes[1].bar(x2 + width/2, df_decades['MAE'], width, label='MAE', color='#DD8452')

axes[1].set_title('Model Errors by Decade', fontsize=14, pad=15, fontweight='bold')
axes[1].set_ylabel('Error Score', fontsize=12)
axes[1].set_xticks(x2)
axes[1].set_xticklabels(df_decades['Category'], fontsize=11)
axes[1].legend()
axes[1].set_ylim(0, 1.4)

# הוספת תוויות נתונים מעל העמודות (עשורים)
for i in range(len(x2)):
    axes[1].text(x2[i] - width/2, df_decades['RMSE'][i] + 0.02, f"{df_decades['RMSE'][i]:.3f}", ha='center', va='bottom', fontsize=10)
    axes[1].text(x2[i] + width/2, df_decades['MAE'][i] + 0.02, f"{df_decades['MAE'][i]:.3f}", ha='center', va='bottom', fontsize=10)

# ==========================================
# 3. הצגת הגרף
# ==========================================
plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# תא 14: Fairness Analysis - כולל השוואה לממוצע הכללי באחוזים
# ==============================================================================
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error

print("⏳ מכין את טבלת הנתונים לניתוח הוגנות והשוואה לממוצע הכללי...")

# 1. חישוב המדדים הגלובליים (על כל הדאטה)
global_rmse = np.sqrt(mean_squared_error(df_fairness['Actual'], df_fairness['RF_Pred']))
global_std = df_fairness['Actual'].std()

print("\n==================================================")
print(" 🌍 מדדי בסיס (Global Metrics)")
print("==================================================")
print(f" - RMSE ממוצע כללי: {global_rmse:.3f}")
print(f" - רעש ממוצע (סטיית תקן של הציונים בפועל): {global_std:.3f}")
print("==================================================\n")

# פונקציית עזר להדפסת הנתונים והפער באחוזים
def print_slice_metrics(slice_name, subset, global_rmse):
    if len(subset) == 0:
        return
        
    slice_rmse = np.sqrt(mean_squared_error(subset['Actual'], subset['RF_Pred']))
    slice_std = subset['Actual'].std()
    
    # החישוב החשוב: כמה החתך רחוק מהממוצע באחוזים
    diff_pct = ((slice_rmse - global_rmse) / global_rmse) * 100
    
    # עיצוב הסימן (+ אם גרוע מהממוצע, - אם טוב מהממוצע)
    sign = "+" if diff_pct > 0 else ""
    
    print(f"🔍 חתך: {slice_name}")
    print(f"   • כמות (n): {len(subset):,}")
    print(f"   • שגיאה (RMSE): {slice_rmse:.3f} ({sign}{diff_pct:.1f}% ביחס לממוצע הכללי)")
    print(f"   • רעש (סטיית תקן הציונים בפועל): {slice_std:.3f}")
    print("-" * 50)


# ---------------------------------------------------------
# א. חתכי ז'אנרים
# ---------------------------------------------------------
print(">>> חתכי ז'אנרים (Top 5) <<<\n")
# פונקציה חכמה לחילוץ ז'אנרים
def extract_genres_to_list(val):
    if pd.isna(val) or str(val).strip() == '': return []
    clean_str = str(val).replace('[', '').replace(']', '').replace("'", "").replace('"', '')
    return [g.strip() for g in clean_str.split(',') if g.strip()]

df_fairness['genres_list'] = df_fairness['genres'].apply(extract_genres_to_list)
df_exploded = df_fairness.explode('genres_list').dropna(subset=['genres_list'])
top_5_genres = df_exploded['genres_list'].value_counts().head(5).index.tolist()

for genre in top_5_genres:
    subset = df_exploded[df_exploded['genres_list'] == genre]
    print_slice_metrics(f"ז'אנר {genre}", subset, global_rmse)


# ---------------------------------------------------------
# ב. חתכי עשורים
# ---------------------------------------------------------
print("\n>>> חתכי עשורים (1990, 2000, 2010) <<<\n")
target_decades = [1990, 2000, 2010]
df_fairness['startYear'] = pd.to_numeric(df_fairness['startYear'], errors='coerce')
df_fairness['Decade'] = (df_fairness['startYear'] // 10) * 10

for decade in target_decades:
    subset = df_fairness[df_fairness['Decade'] == decade]
    print_slice_metrics(f"שנות ה-{decade}", subset, global_rmse)


# ---------------------------------------------------------
# ג. שלמות נתונים (תקציב חסר מול קיים)
# ---------------------------------------------------------
print("\n>>> חתכי שלמות נתונים (תקציב) <<<\n")
df_fairness['budget_clean'] = pd.to_numeric(df_fairness['budget'].astype(str).str.replace(r'[^\d.]', '', regex=True), errors='coerce')

with_budget = df_fairness[df_fairness['budget_clean'].notna()]
missing_budget = df_fairness[df_fairness['budget_clean'].isna()]

print_slice_metrics("יש נתוני תקציב (No Missingness)", with_budget, global_rmse)
print_slice_metrics("אין נתוני תקציב (Missingness)", missing_budget, global_rmse)

In [ ]:
# ==============================================================================
# תא 15: חילוץ Feature Importance וכיוון השפעה לשני המודלים (סעיף 7) - מתוקן
# ==============================================================================
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("⏳ מאמן מודלים ומושך חשיבויות פיצ'רים (ייקח מספר שניות)...")

# 1. אימון המודלים על כל המידע כדי לקבל את המקדמים המדויקים ביותר
rf_pipeline.fit(X, y)
pipeline.fit(X, y)

# 2. חילוץ חכם של שמות העמודות (פותר את ה-AttributeError של Numpy)
temp_df = X.head(2).copy()
feature_names = temp_df.columns.tolist()

# נעבור שלב-שלב ונשמור את השמות מהשלב האחרון שעדיין מתנהג כמו טבלה
for name, step in pipeline[:-1].steps:
    temp_df = step.transform(temp_df)
    if isinstance(temp_df, pd.DataFrame):
        feature_names = temp_df.columns.tolist()

# נעביר את כל הנתונים לעיבוד ונחזיר אותם בחזרה להיות טבלה עם השמות שמצאנו
X_full_transformed_np = pipeline[:-1].transform(X)
X_full_transformed = pd.DataFrame(X_full_transformed_np, columns=feature_names, index=X.index)

# ==========================================
# Elastic Net Analysis (לינארי)
# ==========================================
en_model = pipeline.steps[-1][1]
en_coefs = en_model.coef_

df_en = pd.DataFrame({'Feature': feature_names, 'Coef': en_coefs})
df_en['Abs_Coef'] = df_en['Coef'].abs()
# טופ 5 לפי גודל מוחלט
top_en = df_en.sort_values(by='Abs_Coef', ascending=False).head(5).copy()
top_en['Direction'] = np.where(top_en['Coef'] > 0, 'חיובי (+)', 'שלילי (-)')

# ==========================================
# Random Forest Analysis (עץ)
# ==========================================
rf_model = rf_pipeline.steps[-1][1]
rf_importances = rf_model.feature_importances_

df_rf = pd.DataFrame({'Feature': feature_names, 'Importance': rf_importances})
top_rf = df_rf.sort_values(by='Importance', ascending=False).head(5).copy()

# יער אקראי לא מספק כיוון ישירות. נחשב קורלציה (מתאם פירסון) מול התוצאה
directions = []
for feat in top_rf['Feature']:
    corr = np.corrcoef(X_full_transformed[feat].fillna(0), y)[0, 1]
    directions.append('חיובי (+)' if corr > 0 else 'שלילי (-)')
top_rf['Direction'] = directions

# ==========================================
# הדפסת התוצאות
# ==========================================
print("=======================================================")
print(" 📉 טופ 5 פיצ'רים - מודל לינארי (Elastic Net)")
print("=======================================================")
for idx, row in top_en.iterrows():
    print(f" - {row['Feature']:<25} | מקדם: {row['Coef']:>7.4f} | כיוון השפעה: {row['Direction']}")

print("\n=======================================================")
print(" 🌲 טופ 5 פיצ'רים - מודל עץ (Random Forest)")
print("=======================================================")
for idx, row in top_rf.iterrows():
    print(f" - {row['Feature']:<25} | חשיבות: {row['Importance']:>7.4f} | כיוון השפעה: {row['Direction']}")

In [ ]:
# ==============================================================================
# תא 16: ויזואליזציה של הטיות וחסרונות נתונים (לדוח)
# ==============================================================================
import matplotlib.pyplot as plt
import seaborn as sns

# הגדרת סגנון אקדמי ונקי
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ---------------------------------------------------------
# גרף 1: Informative Missingness (בר-צ'ארט)
# ---------------------------------------------------------
# נתונים מהפלט שלך
categories = ['US Movies', 'Non-US Movies']
missing_rates = [55.2, 92.9]

sns.barplot(x=categories, y=missing_rates, palette=['#4C72B0', '#C44E52'], ax=axes[0])
axes[0].set_title('Informative Missingness: % of Movies Missing Budget', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Percentage Missing (%)', fontsize=12)
axes[0].set_ylim(0, 100)

# הוספת התוויות על העמודות
for i, v in enumerate(missing_rates):
    axes[0].text(i, v + 2, f"{v}%", ha='center', fontsize=12, fontweight='bold')

# ---------------------------------------------------------
# גרף 2: Variance Collapse & Bias (KDE Plot - התפלגות)
# ---------------------------------------------------------
# אנו מציירים את התפלגות הציונים האמיתיים לעומת החזויים עבור הקבוצה *ללא התקציב*
sns.kdeplot(data=missing_budget['Actual'], fill=True, color="blue", label="Actual Ratings (Ground Truth)", ax=axes[1], alpha=0.3)
sns.kdeplot(data=missing_budget['RF_Pred'], fill=True, color="orange", label="Predicted Ratings (Model)", ax=axes[1], alpha=0.5)

axes[1].set_title('Mean Reversion Bias: Actual vs. Predicted (Missing Budget Group)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Average Rating', fontsize=12)
axes[1].set_ylabel('Density', fontsize=12)
axes[1].legend()

# נוסיף קווים שמראים את הממוצע כדי להראות את הקריסה פנימה
axes[1].axvline(missing_budget['Actual'].mean(), color='blue', linestyle='--', alpha=0.7)
axes[1].axvline(missing_budget['RF_Pred'].mean(), color='orange', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# תא 17: חישובי חשיבות (עכשיו רץ על כל הליבות של המחשב כדי לסיים מהר!)
# ==============================================================================
import pandas as pd
import numpy as np
from sklearn.inspection import permutation_importance

print("⏳ מחשב Feature Importance... (רץ מהר עם n_jobs=-1)")

# 1. Elastic Net - חילוץ מקדמים
en_model = pipeline[-1]
coefficients = en_model.coef_

X_temp_en = X.iloc[:2].copy()
for name, step in pipeline.steps:
    if name in ['imputer', 'scaler', 'elasticnet']: break
    X_temp_en = step.transform(X_temp_en)
features_en = X_temp_en.columns

en_importance = pd.DataFrame({
    'Feature': features_en,
    'Coef': coefficients,
    'Abs_Coef': np.abs(coefficients)
}).sort_values(by='Abs_Coef', ascending=False).head(5)

# 2. Random Forest - Gini Importance
rf_model = rf_pipeline[-1]
X_temp_rf = X.iloc[:2].copy()
model_step_name = rf_pipeline.steps[-1][0]
for name, step in rf_pipeline.steps:
    if name in ['imputer', 'scaler', model_step_name]: break
    X_temp_rf = step.transform(X_temp_rf)
processed_features_rf = X_temp_rf.columns

rf_gini_df = pd.DataFrame({
    'Feature': processed_features_rf,
    'Gini_Importance': rf_model.feature_importances_
}).sort_values(by='Gini_Importance', ascending=False).head(5)

# 3. Random Forest - Permutation Importance (מואץ)
X_sample = X.sample(5000, random_state=42) if len(X) > 5000 else X
y_sample = y.loc[X_sample.index]

# כאן הוספתי n_jobs=-1 כדי שזה ירוץ בשניות במקום בשעה
result = permutation_importance(rf_pipeline, X_sample, y_sample, n_repeats=5, random_state=42, scoring='neg_root_mean_squared_error', n_jobs=-1)

rf_perm_df = pd.DataFrame({
    'Feature': X_sample.columns,
    'Perm_Importance': result.importances_mean
}).sort_values(by='Perm_Importance', ascending=False).head(5)

print("✅ החישובים הסתיימו בהצלחה!")

In [ ]:
# ==============================================================================
# תא 18: טבלת מובהקות סטטיסטית באנגלית (Statistical Significance Table)
# ==============================================================================
print("=================================================================================")
print(" Statistical Significance of Permutation Importance")
print("=================================================================================\n")

perm_stats_df = pd.DataFrame({
    'Feature': X_sample.columns,
    'Mean_RMSE_Drop': result.importances_mean,
    'Std_Deviation': result.importances_std
})

perm_stats_df = perm_stats_df[perm_stats_df['Mean_RMSE_Drop'] > 0].sort_values(by='Mean_RMSE_Drop', ascending=False).head(8)

print(f"{'Feature':<25} | {'Mean RMSE Drop (Signal)':<25} | {'Std Deviation (Noise)':<25} | {'SNR':<10}")
print("-" * 95)

for i, row in perm_stats_df.iterrows():
    feat = row['Feature']
    mean_val = row['Mean_RMSE_Drop']
    std_val = row['Std_Deviation']
    snr = mean_val / std_val if std_val > 0 else 0
    print(f"{feat:<25} | {mean_val:<25.4f} | ± {std_val:<23.4f} | {snr:<10.1f}")

print("-" * 95)
print("* Note: SNR (Signal-to-Noise Ratio) = Mean / Std.")
print("* Features with an SNR > 3 are generally considered highly significant and robust.")

In [ ]:
# ==============================================================================
# תא 19: הוכחה ויזואלית להטיה של מדד Gini (Dumbbell Plot)
# ==============================================================================
import matplotlib.pyplot as plt
import seaborn as sns

name_mapping = {
    'genre_target_encoded': 'genres',
    'runtimeMinutes_sq': 'runtimeMinutes',
    'runtimeMinutes_qd': 'runtimeMinutes',
    'cast_mean': 'lead_actors_ids',
    'cast_max': 'lead_actors_ids',
    'cast_min': 'lead_actors_ids'
}

df_gini = rf_gini_df.copy()
df_perm = rf_perm_df.copy()
df_gini['Feature_Mapped'] = df_gini['Feature'].replace(name_mapping)
df_gini['Rel_Gini'] = (df_gini['Gini_Importance'] / df_gini['Gini_Importance'].max()) * 100
df_perm['Rel_Perm'] = (df_perm['Perm_Importance'] / df_perm['Perm_Importance'].max()) * 100

comparison_df = pd.merge(df_perm[['Feature', 'Rel_Perm']], df_gini[['Feature_Mapped', 'Rel_Gini']], left_on='Feature', right_on='Feature_Mapped', how='left').fillna(0)
comparison_df = comparison_df.groupby('Feature', as_index=False).max()

features_to_show = ['genres', 'runtimeMinutes', 'is_decade_2010', 'is_lang_other']
plot_df = comparison_df[comparison_df['Feature'].isin(features_to_show)].set_index('Feature')
plot_df_sorted = plot_df.sort_values(by='Rel_Perm', ascending=True)

sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(12, 7))

ax.hlines(y=plot_df_sorted.index, xmin=plot_df_sorted['Rel_Gini'], xmax=plot_df_sorted['Rel_Perm'], color='grey', alpha=0.4, linewidth=4, zorder=1)
ax.scatter(plot_df_sorted['Rel_Perm'], plot_df_sorted.index, color='#2ca02c', s=350, label='Permutation Importance (Truth)', zorder=2, edgecolors='white', linewidth=2)
ax.scatter(plot_df_sorted['Rel_Gini'], plot_df_sorted.index, color='#d62728', s=120, label='Gini Importance (Biased)', zorder=3, edgecolors='white', linewidth=1)

ax.set_title('The Bias Gap: Permutation vs. MDI (Gini) Importance', fontsize=18, fontweight='bold', pad=20)
ax.set_xlabel('Relative Importance Score (%)', fontsize=14)
ax.set_ylabel('')
ax.tick_params(axis='y', labelsize=14)
ax.tick_params(axis='x', labelsize=12)

for idx, feature in enumerate(plot_df_sorted.index):
    gini_val = plot_df_sorted.loc[feature, 'Rel_Gini']
    if feature in ['is_decade_2010', 'is_lang_other']:
        ax.annotate('Gini scores\nthis as ZERO!', xy=(gini_val, idx), xytext=(gini_val + 12, idx - 0.25), arrowprops=dict(facecolor='#d62728', shrink=0.05, width=1.5, headwidth=7), fontsize=11, color='#d62728', fontweight='bold', bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="#d62728", alpha=0.9))

ax.legend(fontsize=12, loc='lower right', frameon=True, shadow=True, borderpad=1)
plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# תא 20: מחולל טקסט להעתקה ל-Word (שמות, מקדמים וכיוונים)
# ==============================================================================
print("==================================================")
print(" 📝 תוצאות להעתקה והדבקה ישירה לדוח שלכם")
print("==================================================\n")

print("עבור מודל Elastic Net (מודל לינארי):")
for i, row in en_importance.iterrows():
    feat = row['Feature']
    coef = row['Coef']
    direction = "חיובי (+)" if coef > 0 else "שלילי (-)"
    print(f"* {feat} | מקדם: {coef:+.4f} | כיוון: {direction}")

print("\nעבור מודל Random Forest (מודל עצי):")
for i, row in rf_perm_df.iterrows():
    feat = row['Feature']
    imp = row['Perm_Importance']
    
    try:
        corr = np.corrcoef(X_sample[feat].astype(float).fillna(0), y_sample)[0, 1]
        direction = "חיובי (+)" if corr > 0 else "שלילי (-)"
    except:
        direction = "מעורב"
        
    print(f"* {feat} | פגיעה ב-RMSE: {imp:.4f} | כיוון: {direction}")

In [ ]:
# ==============================================================================
# תא 17: השוואת שיטות Feature Importance ליער לעומת Elastic Net
# ==============================================================================
from sklearn.inspection import permutation_importance
import numpy as np
import pandas as pd

print("==================================================")
print(" 🥇 חילוץ חשיבות פיצ'רים - קרב מרובע!")
print("==================================================\n")

# ---------------------------------------------------------
# 1. Elastic Net - חילוץ מקדמים סטנדרטיים (ללא שינוי)
# ---------------------------------------------------------
en_model = pipeline[-1] 
coefficients = en_model.coef_

X_temp = X.iloc[:2].copy()
for name, step in pipeline.steps:
    if name in ['imputer', 'scaler', 'elasticnet']: break 
    X_temp = step.transform(X_temp)
processed_features_en = X_temp.columns 

en_importance = pd.DataFrame({
    'Feature': processed_features_en,
    'Coef': coefficients,
    'Abs_Coef': np.abs(coefficients)
}).sort_values(by='Abs_Coef', ascending=False).head(5)

print(">>> 1. Elastic Net (מקדמים סטנדרטיים) <<<")
for i, row in en_importance.iterrows():
    print(f" {row['Feature']:<25} | מקדם: {row['Coef']:+.4f}")


# ---------------------------------------------------------
# 2. Random Forest - שיטה א': Gini Importance (MDI)
# ---------------------------------------------------------
# נחלץ את המודל עצמו מהפייפליין
rf_model = rf_pipeline[-1]
X_temp_rf = X.iloc[:2].copy()

# >> התיקון הקריטי: עוצרים את הלולאה לפני שהדאטה הופך למערך Numpy חסר שמות <<
# אנחנו בודקים מה השם של המודל האחרון כדי לעצור גם לפניו
model_step_name = rf_pipeline.steps[-1][0] 

for name, step in rf_pipeline.steps:
    # עוצרים בשלבים של Scikit-Learn שמוחקים את שמות העמודות
    if name in ['imputer', 'scaler', model_step_name]: 
        break
    X_temp_rf = step.transform(X_temp_rf)

processed_features_rf = X_temp_rf.columns # עכשיו זה בטוח יעבוד!

gini_importances = rf_model.feature_importances_

rf_gini_df = pd.DataFrame({
    'Feature': processed_features_rf,
    'Gini_Importance': gini_importances
}).sort_values(by='Gini_Importance', ascending=False).head(5)

print("\n>>> 2A. Random Forest (MDI / Gini Importance) - השיטה המובנית <<<")
for i, row in rf_gini_df.iterrows():
    print(f" {row['Feature']:<25} | חשיבות Gini: {row['Gini_Importance']:.4f}")


# ---------------------------------------------------------
# 3. Random Forest - שיטה ב': Permutation Importance
# ---------------------------------------------------------
print("\n⏳ מחשב Permutation Importance (זהב) ליער האקראי...")
X_sample = X.sample(5000, random_state=42) if len(X) > 5000 else X
y_sample = y.loc[X_sample.index]

result = permutation_importance(rf_pipeline, X_sample, y_sample, n_repeats=5, random_state=42, scoring='neg_root_mean_squared_error')

rf_perm_df = pd.DataFrame({
    'Feature': X_sample.columns, 
    'Perm_Importance': result.importances_mean
}).sort_values(by='Perm_Importance', ascending=False).head(5)

print("\n>>> 2B. Random Forest (Permutation Importance) - הסטנדרט לאמינות <<<")
for i, row in rf_perm_df.iterrows():
    print(f" {row['Feature']:<25} | פגיעה ב-RMSE: {row['Perm_Importance']:.4f}")

print("\n==================================================")

In [ ]:
# ==============================================================================
# תא 12: ניתוח שגיאות מתקדם (Levene Test) - שפה אנגלית ומדינה חסרה
# ==============================================================================
import scipy.stats as stats
import pandas as pd
import numpy as np

print("📊 מפיק טבלת ניתוח שגיאות עבור המאפיינים הנבחרים לדוח...")

# 1. הגדרת המנצח (Random Forest מול Elastic Net)
conditions = [
    (df_comp['EN_Abs_Error'] > df_comp['RF_Abs_Error'] + 0.5), # RF טעה פחות
    (df_comp['RF_Abs_Error'] > df_comp['EN_Abs_Error'] + 0.5)  # EN טעה פחות
]
choices = ['RF_Won', 'EN_Won']
df_comp['Winner'] = np.select(conditions, choices, default='Tie')

# 2. העברת הנתונים דרך הפייפליין לטובת חישובים
X_transformed = pipeline[:-1].transform(X)

# חילוץ שמות בטוח למניעת קריסות
try:
    feature_names = list(pipeline[:-1].get_feature_names_out())
except:
    orig_cols = list(X.columns)
    num_features = X_transformed.shape[1]
    num_orig = len(orig_cols)
    if num_features == num_orig:
        feature_names = orig_cols
    else:
        feature_names = orig_cols + [f"Engineered_Feature_{i}" for i in range(num_orig, num_features)]

# הרכבת הנתונים מחדש
df_features = pd.DataFrame(X_transformed, columns=feature_names, index=X.index)
df_features['Winner'] = df_comp['Winner']

# סינון רק לסרטים בהם יש מנצח ברור
df_winners = df_features[df_features['Winner'].isin(['RF_Won', 'EN_Won'])]

# ==========================================
# 3. רשימת העמודות המסוננת (אנגלית ומדינה לא ידועה)
# ==========================================
selected_features = ['unknown_country', 'is_lang_English']

results = []
for col in selected_features:
    # מציאת השם המדויק גם אם הפייפליין הוסיף קידומת
    matching_cols = [c for c in df_winners.columns if col in c]
    
    if len(matching_cols) > 0:
        actual_col_name = matching_cols[0]
        rf_data = df_winners[df_winners['Winner'] == 'RF_Won'][actual_col_name]
        en_data = df_winners[df_winners['Winner'] == 'EN_Won'][actual_col_name]
        
        # וידוא שונות מינימלית
        if rf_data.std() > 0.001 and en_data.std() > 0.001:
            stat, p_val = stats.levene(rf_data, en_data, center='median')
            
            # הוספה לטבלה עם שמות מעוצבים וברורים לדוח
            results.append({
                'Feature (מאפיין)': col,
                'Std (RF Won)': rf_data.std(),
                'Std (EN Won)': en_data.std(),
                'P-Value': p_val
            })
    else:
        print(f"⚠️ שגיאה: העמודה {col} חסרה בנתונים.")

# 4. הדפסה יפה ומסודרת של הטבלה
final_df = pd.DataFrame(results)

print("\n✅ טבלת מובהקות סטטיסטית (Levene Test) - מוכנה להעתקה לדוח:\n")
display(final_df.round(4))